# Simulated XRF spectrum

Runs the model with `configs/lab_xrf.yml` and shows, for the first sample in
`samples`, the strongest lines and the detector spectrum.

`lab_xrf.yml` uses files kept only on this computer: the Geant4 response matrix
and the MCA calibration. On another computer, remove the `detector_response`,
`mca_calibration` and `mca_channels` keys first.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
print("ROOT:", ROOT)

In [ ]:
from xrf_model.config import load_config
from xrf_model.pipeline import run_one_grid

cfg = load_config("configs/lab_xrf.yml")

# lab_xrf.yml has a `tube:` block, so the tube spectrum is generated with
# SpekPy and no flux file is passed. For a solar or measured spectrum,
# use a config without `tube:` and pass the file path instead of None.
res_one = run_one_grid(None, cfg)
res_one.keys()

In [ ]:
sample = cfg["samples"][0]
r = res_one[sample]

# Strongest lines reaching the detector face [ph/s]
r["fluorescence"].sort_values("Intensity_ph_s", ascending=False).head(15)

In [ ]:
import matplotlib.pyplot as plt

# Detector spectrum as computed by the pipeline: the same one written to the
# .xlsx, .dat and .mca files (response matrix, broadening and Compton included).
fig, ax = plt.subplots(figsize=(11, 4))
ax.semilogy(r["spectrum_energy"], r["spectrum"], lw=0.8)
ax.set_xlim(1, 25)
ax.set_ylim(bottom=r["spectrum"].max() * 1e-6)
ax.set_xlabel("Energy (keV)")
ax.set_ylabel("ph/s/keV")
ax.set_title(f"Simulated detector spectrum — {sample}")
ax.grid(True, alpha=0.3)
plt.show()

print("Labelled plot and all output files in:", r["out_dir"])